In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- CRITICAL: BASE PATH DEFINITION ---
# This is the path to the 'Data' folder containing 'train' and 'test'.
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/Data'

# --- YOUR REQUESTED PATH VARIABLES ---

# Training Paths
train_pos_path = os.path.join(BASE_DIR, 'train', 'pos')
train_neg_path = os.path.join(BASE_DIR, 'train', 'neg')

# Testing Paths
test_pos_path = os.path.join(BASE_DIR, 'test', 'pos')
test_neg_path = os.path.join(BASE_DIR, 'test', 'neg')


# --- VERIFICATION ---
print("✅ Path variables defined successfully:")
print(f"train_pos_path: {train_pos_path}")
print(f"train_neg_path: {train_neg_path}")
print(f"test_pos_path: {test_pos_path}")
print(f"test_neg_path: {test_neg_path}")

# Run a quick check to ensure the directories actually exist
if os.path.isdir(train_pos_path):
    print("\nDirectory check successful. Ready to load data.")
else:
    print("\n❌ ERROR: Directory not found. Please confirm the BASE_DIR is correct.")

✅ Path variables defined successfully:
train_pos_path: /content/drive/MyDrive/Colab Notebooks/Data/train/pos
train_neg_path: /content/drive/MyDrive/Colab Notebooks/Data/train/neg
test_pos_path: /content/drive/MyDrive/Colab Notebooks/Data/test/pos
test_neg_path: /content/drive/MyDrive/Colab Notebooks/Data/test/neg

Directory check successful. Ready to load data.


In [ ]:
def load_reviews_from_folder(folder, sentiment):
    reviews = []
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            with open(os.path.join(folder, filename), "r", encoding="utf-8") as f:
                reviews.append(f.read())
    return pd.DataFrame({"review": reviews, "sentiment": sentiment})

In [ ]:
df_train_pos = load_reviews_from_folder(train_pos_path, "positive")
df_train_neg = load_reviews_from_folder(train_neg_path, "negative")

df_train = pd.concat([df_train_pos, df_train_neg]).reset_index(drop=True)

In [ ]:
df_test_pos = load_reviews_from_folder(test_pos_path, "positive")
df_test_neg = load_reviews_from_folder(test_neg_path, "negative")

df_test = pd.concat([df_test_pos, df_test_neg]).reset_index(drop=True)

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# ============================================
# 1. DATA PREPROCESSING
# ============================================
print("📊 Preprocessing data...")

# Convert sentiment to binary labels
df_train['label'] = (df_train['sentiment'] == 'positive').astype(int)
df_test['label'] = (df_test['sentiment'] == 'positive').astype(int)

# Extract texts and labels
X_train_full = df_train['review'].values
y_train_full = df_train['label'].values
X_test = df_test['review'].values
y_test = df_test['label'].values

# Create validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# ============================================
# 2. TOKENIZATION
# ============================================
print("\n🔤 Tokenizing text...")

# Use top 20,000 most frequent words
MAX_WORDS = 20000
MAX_LEN = 250  # Maximum sequence length

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert texts to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Vocabulary size: {len(tokenizer.word_index) + 1}")
print(f"Sequence shape: {X_train_pad.shape}")

In [ ]:
# ============================================
# 3. ATTENTION LAYER (Custom)
# ============================================
class AttentionLayer(layers.Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], input_shape[-1]),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[-1],),
            initializer='zeros',
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        # x shape: (batch_size, time_steps, hidden_size)
        e = tf.nn.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)

In [ ]:
# ============================================
# 4. BUILD ADVANCED RNN MODEL
# ============================================
print("\n🏗️ Building Advanced Bi-LSTM with Attention...")

def build_advanced_rnn():
    model = keras.Sequential([
        # Embedding Layer with pre-trained style initialization
        layers.Embedding(
            input_dim=MAX_WORDS,
            output_dim=128,
            input_length=MAX_LEN,
            embeddings_initializer='glorot_uniform',
            mask_zero=True
        ),

        # Spatial Dropout for regularization
        layers.SpatialDropout1D(0.3),

        # First Bidirectional LSTM Layer
        layers.Bidirectional(
            layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)
        ),

        # Second Bidirectional LSTM Layer
        layers.Bidirectional(
            layers.LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)
        ),

        # Attention Mechanism
        AttentionLayer(),

        # Dense layers with BatchNormalization
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),

        # Output layer
        layers.Dense(1, activation='sigmoid')
    ])

    return model

model = build_advanced_rnn()

# Compile with Adam optimizer and learning rate schedule
initial_learning_rate = 0.001
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=1000,
    decay_rate=0.96,
    staircase=True
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

print(model.summary())

In [ ]:
# ============================================
# 5. CALLBACKS
# ============================================
callbacks = [
    # Early stopping
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate on plateau
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    # Model checkpoint
    keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
# ============================================
# 6. TRAIN THE MODEL
# ============================================
print("\n🚀 Training the model...")

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ============================================
# 7. EVALUATE ON TEST SET
# ============================================
print("\n📈 Evaluating on test set...")

test_loss, test_acc, test_precision, test_recall = model.evaluate(X_test_pad, y_test, verbose=1)

print(f"\n{'='*50}")
print(f"🎯 TEST RESULTS:")
print(f"{'='*50}")
print(f"Accuracy:  {test_acc*100:.2f}%")
print(f"Precision: {test_precision*100:.2f}%")
print(f"Recall:    {test_recall*100:.2f}%")
print(f"F1-Score:  {2*(test_precision*test_recall)/(test_precision+test_recall)*100:.2f}%")
print(f"{'='*50}")

In [ ]:
# ============================================
# 8. PLOT TRAINING HISTORY
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 9. PREDICTION FUNCTION
# ============================================
def predict_sentiment(text):
    """Predict sentiment of a single review"""
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=MAX_LEN, padding='post', truncating='post')
    prediction = model.predict(padded, verbose=0)[0][0]

    sentiment = "POSITIVE ✅" if prediction > 0.5 else "NEGATIVE ❌"
    confidence = prediction if prediction > 0.5 else 1 - prediction

    print(f"\nReview: {text[:100]}...")
    print(f"Sentiment: {sentiment}")
    print(f"Confidence: {confidence*100:.2f}%")

    return prediction

# Test with sample reviews
print("\n" + "="*50)
print("🧪 SAMPLE PREDICTIONS:")
print("="*50)

sample_reviews = [
    "This movie was absolutely amazing! Best film I've seen this year.",
    "Terrible waste of time. Poor acting and boring plot.",
    "It was okay, nothing special but not bad either."
]

for review in sample_reviews:
    predict_sentiment(review)